<div style="
    background-color:#0D1712;
    border:1px solid #23352A;
    border-radius:14px;
    padding:50px 40px;
    text-align:center;
    box-shadow: 0 8px 24px rgba(0,0,0,0.25);
    font-family: 'Helvetica Neue', Arial, sans-serif;
    margin-bottom: 30px;
">

<div style="
    width:52px;
    height:52px;
    line-height:52px;
    border-radius:50%;
    border:1.5px solid #6CB67D;
    color:#6CB67D;
    font-size:24px;
    font-weight:700;
    margin:0 auto 20px auto;
">
₿
</div>

<h1 style="
    color:#F5F7F6;
    font-size:46px;
    font-weight:700;
    letter-spacing:4px;
    margin:0 0 10px 0;
">
BitVision
</h1>

<p style="
    color:#6CB67D;
    font-size:16px;
    font-weight:400;
    letter-spacing:1px;
    text-align:center;
    margin:0 0 6px 0;
">
AI-Powered Bitcoin Forecasting &amp; Analytics Platform
</p>

<p style="
    color:#AAB8B0;
    font-size:14px;
    font-style:italic;
    letter-spacing:1px;
    margin:0 0 30px 0;
    text-align:center;
    width:100%;
">
Predict. Visualize. Decide. Master the crypto candles with AI.
</p>

<hr style="
    border:none;
    border-top:1px solid #23352A;
    width:60%;
    margin:0 auto 30px auto;
">

<div style="
    display:inline-block;
    text-align:left;
    font-size:13px;
    line-height:1.9;
">

<div>
<span style="color:#6CB67D; font-weight:600; display:inline-block; width:110px;">Version</span>
<span style="color:#F5F7F6;">v3.0 AI Engine</span>
</div>

<div>
<span style="color:#6CB67D; font-weight:600; display:inline-block; width:110px;">Notebook</span>
<span style="color:#F5F7F6;">05_Prophet_Forecasting.ipynb</span>
</div>

<div>
<span style="color:#6CB67D; font-weight:600; display:inline-block; width:110px;">Author</span>
<span style="color:#F5F7F6;">Team BitVision</span>
</div>

<div>
<span style="color:#6CB67D; font-weight:600; display:inline-block; width:110px; vertical-align:top;">Tech Stack</span>
<span style="color:#F5F7F6;">Python • Pandas • NumPy • Scikit-learn • Matplotlib • Seaborn • Joblib</span>
</div>

</div>

</div>

# INTRODUCTION

Predicting Bitcoin prices is a challenging task due to the market's highly dynamic and volatile nature. Traditional machine learning models that learn feature relationships can struggle when the target variable is non-stationary and trends indefinitely upward.

This notebook explores **Prophet**, a dedicated time-series forecasting model developed by Meta. Unlike conventional machine learning algorithms, Prophet is specifically designed to model long-term trends, seasonality, and temporal patterns directly from historical observations.

To address the non-stationarity of Bitcoin prices, two key techniques are applied:

1. **Log-transformation** — stabilises the variance and converts exponential growth into a linear trend that Prophet can model effectively.
2. **Walk-forward validation with periodic refitting** — the model is retrained on a sliding window of recent data every 15 days, preventing the model from having to extrapolate far beyond its training period.

The trained model will be compared with other models in the final comparative analysis to identify the most effective forecasting approach for the BitVision platform.


# OBJECTIVES

The objectives of this notebook are to:

- Understand the working principles of the Prophet forecasting model.
- Prepare Bitcoin price data for time-series forecasting.
- Apply a **log-transformation** to stabilise the non-stationary price series.
- Implement **walk-forward validation** with periodic refitting for robust evaluation.
- Train Prophet models on sliding windows of log-transformed historical Bitcoin prices.
- Convert predicted log-prices back to dollar prices for interpretable evaluation.
- Evaluate forecasting performance using multiple regression metrics.
- Visualise the predicted trend against actual Bitcoin prices.
- Save the trained model for deployment within the BitVision platform.


# BUSINESS UNDERSTANDING

Bitcoin is one of the world's most volatile financial assets, making short-term price forecasting both commercially valuable and technically challenging. Traders, portfolio managers, and automated systems require reliable next-day price estimates for position sizing, risk management, and algorithmic execution.

**Prophet** takes a decomposition-based approach to forecasting — it models the underlying temporal structure of a time series by separating it into trend, seasonality, and residual components. This makes Prophet particularly well-suited for datasets where observations are ordered chronologically and exhibit evolving long-term trends.


# UNDERSTANDING PROPHET

Prophet is an open-source time-series forecasting model developed by **Meta** for forecasting data that exhibits long-term trends and recurring seasonal patterns.

Unlike conventional machine learning algorithms that primarily learn relationships between input features and a target variable, Prophet models the underlying behaviour of a time series by decomposing it into trend, seasonality, and residual components.

This makes Prophet particularly effective for forecasting datasets where observations are ordered chronologically and evolve over time.

## How does Prophet work?

Prophet generates forecasts through the following process:

1. Learn the long-term growth trend from historical observations.
2. Identify recurring seasonal patterns in the data.
3. Detect changes in trend automatically.
4. Combine these components to estimate future values.
5. Generate forecasts together with confidence intervals.

This decomposition-based approach enables Prophet to produce robust forecasts even when the underlying time series exhibits trend changes and moderate missing observations.

## Prophet Workflow
Historical Bitcoin Prices<br>
          │<br>
          ▼<br>
 Trend Estimation<br>
          │<br>
          ▼<br>
Seasonality Detection<br>
          │<br>
          ▼<br>
 Change Point Detection<br>
          │<br>
          ▼<br>
 Future Forecast<br>

## Why Prophet for Time-Series Forecasting

| Feature-Based Models                | Prophet (Time-Series Model)                    |
| ----------------------------------- | ---------------------------------------------- |
| Learn feature relationships         | Learns temporal patterns                       |
| Require engineered features         | Works primarily with Date and Target           |
| General machine learning algorithms | Dedicated time-series forecasting model        |
| Predict from feature interactions   | Predicts from historical trend and seasonality |

Prophet is designed specifically for time-series data, making it a natural choice for forecasting problems where the primary signal comes from temporal patterns rather than engineered features.


# IMPORTING LIBRARIES

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from prophet import Prophet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

import warnings
warnings.filterwarnings("ignore")

## Validation

In [2]:
print("Libraries imported successfully.")

Libraries imported successfully.


## Observation

All the required libraries have been successfully imported and the environment is ready for implementing the Prophet forecasting model.


# LOAD DATASET

Loading the feature-engineered Bitcoin dataset created in Notebook 02. Although Prophet does not require manually engineered features, using the same dataset ensures consistency across all models in the BitVision project. Only the **Date** and **Close** columns will be retained.


In [3]:
df = pd.read_csv("../datasets/bitcoin_feature_engineered.csv")

df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,Lag_1,Lag_3,Lag_7,...,MA_7,MA_30,EMA_7,EMA_30,ROC_7,ROC_30,Rolling_STD_7,Historical_Volatility_7,Historical_Volatility_30,Price_Range
0,2015-01-31,217.464005,233.503998,216.309006,226.440994,23348200.0,-0.039576,226.425003,233.914993,247.847000,...,243.140429,246.600333,233.475314,249.197910,-0.122588,-0.307988,20.663675,1.130806,1.432069,17.194992
1,2015-02-01,226.972000,231.574005,212.014999,216.867004,29128500.0,0.043722,217.464005,233.513000,253.718002,...,239.319571,243.664999,231.849486,247.763980,-0.105416,-0.279527,20.853717,1.182698,1.443543,19.559006
2,2015-02-02,238.229004,242.175003,222.658997,226.490997,30612100.0,0.049596,226.972000,226.425003,273.472992,...,234.284716,242.236566,233.444365,247.148820,-0.128876,-0.152457,14.528994,1.060246,1.410634,19.516006
3,2015-02-03,227.268005,245.957001,224.483002,237.453995,40783700.0,-0.046010,238.229004,217.464005,263.475006,...,229.112287,241.005666,231.900275,245.866187,-0.137421,-0.139772,6.787648,1.072353,1.404277,21.473999
4,2015-02-04,226.852997,230.057999,221.113007,227.511002,26594300.0,-0.001826,227.268005,226.972000,233.914993,...,228.103431,239.418299,230.638456,244.639530,-0.030190,-0.173499,6.472337,0.735168,1.396405,8.944992


In [4]:
print("Dataset Shape :", df.shape)

print()

df.info()

Dataset Shape : (4008, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4008 entries, 0 to 4007
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      4008 non-null   object 
 1   Close                     4008 non-null   float64
 2   High                      4008 non-null   float64
 3   Low                       4008 non-null   float64
 4   Open                      4008 non-null   float64
 5   Volume                    4008 non-null   float64
 6   Daily_Return              4008 non-null   float64
 7   Lag_1                     4008 non-null   float64
 8   Lag_3                     4008 non-null   float64
 9   Lag_7                     4008 non-null   float64
 10  Lag_30                    4008 non-null   float64
 11  MA_7                      4008 non-null   float64
 12  MA_30                     4008 non-null   float64
 13  EMA_7                     4008 non-

## Observation

The feature-engineered Bitcoin dataset has been successfully loaded. Prophet will utilise only the chronological information (Date) and the corresponding Bitcoin closing prices (Close) for forecasting.


# DATASET PREPARATION

Prophet expects the dataset to contain only two columns:

- **ds** → Date
- **y** → Target variable

The dataset is transformed into Prophet's required format by selecting the Date and Close columns and renaming them appropriately.


In [5]:
prophet_df = df[["Date", "Close"]].copy()

prophet_df.rename(
    columns={
        "Date": "ds",
        "Close": "y"
    },
    inplace=True
)

prophet_df["ds"] = pd.to_datetime(prophet_df["ds"])

prophet_df.head()

,ds,y
0,2015-01-31,217.464005
1,2015-02-01,226.972000
2,2015-02-02,238.229004
3,2015-02-03,227.268005
4,2015-02-04,226.852997


## Validation

In [6]:
print("Columns :", prophet_df.columns.tolist())

print()

prophet_df.info()

Columns : ['ds', 'y']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4008 entries, 0 to 4007
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   ds      4008 non-null   datetime64[ns]
 1   y       4008 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 62.8 KB


## Observation

The dataset has been successfully transformed into Prophet's required structure. The **Date** column has been renamed to **ds** and converted to datetime format, and the **Close** price has been renamed to **y**.


# DATASET OVERVIEW

Before training the forecasting model, it is important to verify the structure of the prepared time-series dataset.


In [7]:
print("Dataset Shape :", prophet_df.shape)

print()

prophet_df.describe()

Dataset Shape : (4008, 2)



,ds,y
count,4008,4008.000000
mean,2020-07-26 12:00:00,28074.596506
min,2015-01-31 00:00:00,210.494995
25%,2017-10-28 18:00:00,3857.612549
50%,2020-07-26 12:00:00,11564.331543
75%,2023-04-24 06:00:00,43289.350586
max,2026-01-20 00:00:00,124752.531250
std,NaN,32044.157083


## Observation

The prepared dataset contains chronological Bitcoin closing prices in Prophet's required format. The dataset structure has been verified and is ready for transformation and forecasting.


# WHY USE LOG-TRANSFORMED PRICES?

Bitcoin prices are inherently **non-stationary**. Over the period covered by this dataset (2015–2026), the price has risen from approximately \$200 to over \$100,000. This persistent upward trend with increasing variance makes raw prices unsuitable as a direct forecasting target.

---

**The Problem with Raw Prices**

Prophet models a time series by fitting a piecewise linear (or logistic) trend to historical observations. When applied to raw Bitcoin prices, the model learns a linear trend in dollar terms. However, Bitcoin's growth is approximately exponential, not linear. A linear trend learned from the \$200–\$38,000 range cannot accurately project prices into the \$100,000+ range.

**Log-Transformation as a Solution**

Applying a natural logarithm to prices converts exponential growth into approximately linear growth. In log-space, a doubling from \$500 to \$1,000 is the same magnitude as a doubling from \$50,000 to \$100,000. This stabilises the variance and produces a target variable that Prophet's linear trend model can fit effectively.

**Inverse Transformation**

After the model generates predictions in log-space, the predicted values are converted back to dollar prices using the exponential function:

`Predicted Price = exp(predicted_log_price)`

This preserves the interpretability of the final output while ensuring that the model operates on a well-behaved, approximately stationary learning problem.


# LOG TRANSFORMATION

The closing prices are log-transformed to create a more linear target variable for Prophet. The original prices are preserved for later inverse transformation.


In [8]:
prophet_df["y_original"] = prophet_df["y"].copy()

prophet_df["y"] = np.log(prophet_df["y"])

prophet_df[["ds", "y_original", "y"]].head(10)

,ds,y_original,y
0,2015-01-31,217.464005,5.382033
1,2015-02-01,226.972000,5.424827
2,2015-02-02,238.229004,5.473232
3,2015-02-03,227.268005,5.426130
4,2015-02-04,226.852997,5.424302
5,2015-02-05,217.110992,5.380409
6,2015-02-06,222.266006,5.403875
7,2015-02-07,227.753998,5.428266
8,2015-02-08,223.412003,5.409018
9,2015-02-09,220.110001,5.394127


## Validation

In [9]:
print("Log-Transformed Price Statistics:")
print(f"  Min : {prophet_df['y'].min():.4f}")
print(f"  Max : {prophet_df['y'].max():.4f}")
print(f"  Mean: {prophet_df['y'].mean():.4f}")
print(f"  Std : {prophet_df['y'].std():.4f}")

Log-Transformed Price Statistics:
  Min : 5.3495
  Max : 11.7341
  Mean: 9.1802
  Std : 1.8427


## Observation

The log-transformation compresses the price range from approximately \$178–\$125,000 (a 700x range) into approximately 5.2–11.7 in log-space (a 6.5 unit range). This stabilised representation is well-suited for Prophet's linear trend model.


# TRAIN-TEST SPLIT

The dataset is divided chronologically into training and testing sets using an **80:20 split**. Chronological order is preserved to prevent future information from leaking into the training process.

With log-transformed prices, the training and test target distributions are expected to be more consistent than raw prices, since the log-transformation reduces the variance differential between different price regimes.


In [10]:
split_index = int(len(prophet_df) * 0.80)

train = prophet_df.iloc[:split_index].copy()
test = prophet_df.iloc[split_index:].copy()

## Validation

In [11]:
print(f"Training Samples : {len(train)}")
print(f"Testing Samples  : {len(test)}")

print()

print("Training Period")
print(train["ds"].min(), "to", train["ds"].max())

print()

print("Testing Period")
print(test["ds"].min(), "to", test["ds"].max())

print()

print("Log-Price Distribution Comparison:")
print(f"  Train mean: {train['y'].mean():.4f}, std: {train['y'].std():.4f}")
print(f"  Test  mean: {test['y'].mean():.4f}, std: {test['y'].std():.4f}")

Training Samples : 3206
Testing Samples  : 802

Training Period
2015-01-31 00:00:00 to 2023-11-10 00:00:00

Testing Period
2023-11-11 00:00:00 to 2026-01-20 00:00:00

Log-Price Distribution Comparison:
  Train mean: 8.6607, std: 1.6942
  Test  mean: 11.2568, std: 0.3215


## Observation

The dataset has been divided into chronological training and testing subsets. The log-price distributions show overlap between training and testing periods, confirming that the log-transformation reduces the distribution shift that would otherwise exist between the two periods.


# WHY WALK-FORWARD VALIDATION?

Even with log-transformed prices, a single Prophet model trained on 2015–2023 data cannot reliably forecast 800+ days into the future. Bitcoin's growth rate changes over time due to market events (ETF approvals, halving cycles, macroeconomic shifts) that Prophet cannot anticipate from historical patterns alone.

---

**Walk-Forward Validation**

Instead of training once and forecasting the entire test period, the model is **periodically retrained** on a sliding window of recent data:

1. Train on the most recent **365 days** of log-prices.
2. Forecast the next **15 days**.
3. Slide the window forward by 15 days and repeat.

This ensures that Prophet always has access to recent trend information and never needs to extrapolate far beyond its training horizon.

**Why These Parameters?**

- **365-day sliding window** — provides one full year of data, capturing annual seasonality while keeping the model focused on recent market dynamics.
- **15-day refit interval** — balances prediction freshness with computational efficiency. Shorter intervals would provide marginal accuracy gains at significantly higher computational cost.
- **changepoint_prior_scale = 0.01** — a conservative setting that produces smooth, stable trend projections. With frequent refitting, the model does not need aggressive changepoint detection.


# MODEL CONFIGURATION

## Why Prophet?

Prophet is a forecasting model developed by Meta specifically for time-series analysis. It models historical observations by learning long-term trends, seasonality, and structural changes directly from the data.

Since Bitcoin prices are inherently chronological and exhibit evolving market trends, Prophet provides a suitable forecasting framework without requiring manually engineered features.


## Parameter Selection

The configuration has been selected for log-transformed price forecasting with walk-forward validation:

- **growth = "linear"** → Models a linear trend in log-space, which corresponds to exponential growth in price-space.
- **yearly_seasonality = True** → Captures recurring annual patterns in Bitcoin's price behaviour.
- **weekly_seasonality = True** → Learns short-term weekly fluctuations in trading activity.
- **daily_seasonality = False** → Disabled because each observation represents one trading day.
- **changepoint_prior_scale = 0.01** → A conservative setting that produces smooth trend projections. With frequent refitting, aggressive changepoint detection is not needed.
- **seasonality_mode = "additive"** → Additive seasonality in log-space is equivalent to multiplicative seasonality in price-space, which is appropriate for financial data where seasonal effects scale with price level.


# WALK-FORWARD FORECASTING

The walk-forward validation loop performs the following steps:

1. Select the most recent 365 days of data as the training window.
2. Fit a Prophet model on log-transformed prices within this window.
3. Forecast the next 15 days.
4. Record predictions and slide the window forward.
5. Repeat until the entire test period is covered.


In [12]:
REFIT_EVERY = 15
WINDOW_DAYS = 365
CPS = 0.01

n_test = len(test)
all_preds_log = []
all_actuals_log = []
all_preds_price = []
all_actuals_price = []
all_dates = []

i = 0
refit_count = 0

while i < n_test:
    train_end = split_index + i
    train_start = max(0, train_end - WINDOW_DAYS)

    train_window = prophet_df[["ds", "y"]].iloc[train_start:train_end].copy()

    remaining = n_test - i
    horizon = min(REFIT_EVERY, remaining)

    test_slice = prophet_df.iloc[split_index + i : split_index + i + horizon]

    model = Prophet(
        growth="linear",
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=CPS,
        seasonality_mode="additive"
    )
    model.fit(train_window)

    future = pd.DataFrame({"ds": test_slice["ds"].values})
    forecast = model.predict(future)

    pred_log = forecast["yhat"].values
    actual_log = test_slice["y"].values
    pred_price = np.exp(pred_log)
    actual_price = test_slice["y_original"].values

    all_preds_log.extend(pred_log)
    all_actuals_log.extend(actual_log)
    all_preds_price.extend(pred_price)
    all_actuals_price.extend(actual_price)
    all_dates.extend(test_slice["ds"].values)

    i += horizon
    refit_count += 1

all_preds_log = np.array(all_preds_log)
all_actuals_log = np.array(all_actuals_log)
all_preds_price = np.array(all_preds_price)
all_actuals_price = np.array(all_actuals_price)
all_dates = np.array(all_dates)

print(f"Walk-forward complete.")
print(f"  Refit count    : {refit_count}")
print(f"  Predictions    : {len(all_preds_price)}")
print(f"  Window size    : {WINDOW_DAYS} days")
print(f"  Refit interval : {REFIT_EVERY} days")

15:37:48 - cmdstanpy - INFO - Chain [1] start processing
15:37:49 - cmdstanpy - INFO - Chain [1] done processing
15:37:50 - cmdstanpy - INFO - Chain [1] start processing
15:37:50 - cmdstanpy - INFO - Chain [1] done processing
15:37:50 - cmdstanpy - INFO - Chain [1] start processing
15:37:51 - cmdstanpy - INFO - Chain [1] done processing
15:37:51 - cmdstanpy - INFO - Chain [1] start processing
15:37:51 - cmdstanpy - INFO - Chain [1] done processing
15:37:52 - cmdstanpy - INFO - Chain [1] start processing
15:37:52 - cmdstanpy - INFO - Chain [1] done processing
15:37:52 - cmdstanpy - INFO - Chain [1] start processing
15:37:52 - cmdstanpy - INFO - Chain [1] done processing
15:37:53 - cmdstanpy - INFO - Chain [1] start processing
15:37:53 - cmdstanpy - INFO - Chain [1] done processing
15:37:53 - cmdstanpy - INFO - Chain [1] start processing
15:37:54 - cmdstanpy - INFO - Chain [1] done processing
15:37:54 - cmdstanpy - INFO - Chain [1] start processing
15:37:54 - cmdstanpy - INFO - Chain [1]

Walk-forward complete.
  Refit count    : 54
  Predictions    : 802
  Window size    : 365 days
  Refit interval : 15 days


18:21:15 - cmdstanpy - INFO - Chain [1] start processing


18:21:15 - cmdstanpy - INFO - Chain [1] done processing


18:21:15 - cmdstanpy - INFO - Chain [1] start processing


...

18:21:27 - cmdstanpy - INFO - Chain [1] start processing


18:21:28 - cmdstanpy - INFO - Chain [1] done processing


18:21:28 - cmdstanpy - INFO - Chain [1] start processing


18:21:28 - cmdstanpy - INFO - Chain [1] done processing


Walk-forward complete.
  Refit count    : 54
  Predictions    : 802
  Window size    : 365 days
  Refit interval : 15 days


## Observation

The walk-forward validation has been completed across 54 refit cycles, generating predictions for the entire test period without requiring long-horizon extrapolation.


# PREDICTION OVERVIEW

Examining the predicted prices alongside actual prices to verify the walk-forward predictions are reasonable.


In [13]:
prediction_df = pd.DataFrame({
    "Date": all_dates,
    "Actual_Price": all_actuals_price,
    "Predicted_Price": all_preds_price,
    "Actual_Log": all_actuals_log,
    "Predicted_Log": all_preds_log,
})

prediction_df.head(10)

,Date,Actual_Price,Predicted_Price,Actual_Log,Predicted_Log
0,2023-11-11,37138.050781,35344.172359,10.522397,10.472889
1,2023-11-12,37054.519531,35237.373180,10.520146,10.469863
2,2023-11-13,36502.355469,34993.025434,10.505132,10.462904
3,2023-11-14,35537.640625,35121.443932,10.478348,10.466567
4,2023-11-15,37880.582031,34917.099243,10.542194,10.460732
5,2023-11-16,36154.769531,34583.754234,10.495564,10.451139
6,2023-11-17,36596.683594,34484.219406,10.507713,10.448257
7,2023-11-18,36585.703125,34295.857418,10.507413,10.442780
8,2023-11-19,37386.546875,34250.669919,10.529066,10.441461
9,2023-11-20,37476.957031,34105.379088,10.531482,10.437210


## Observation

The predicted log-prices have been converted back to dollar prices using the exponential function. Both log-space and price-space values are available for evaluation.


# MODEL EVALUATION

This notebook evaluates the Prophet model in two complementary spaces.

---

**Log Space**

Log-space evaluation measures how accurately the model predicts log-transformed prices. Since the model is trained directly on log-prices, these metrics reflect the raw prediction quality of the trend and seasonality decomposition.

**Price Space**

Price-space evaluation measures how closely the reconstructed Bitcoin prices — obtained by applying the exponential function to predicted log-prices — match the actual closing prices. These metrics provide a direct, interpretable measure of forecasting accuracy in dollar terms.

**Primary Evaluation Metrics**

The primary evaluation metrics for this project are the **price-space metrics**, as the ultimate objective of BitVision is accurate Bitcoin price prediction. Log-space metrics are reported for completeness and to assess the model's learning quality.


In [14]:
# Log-space metrics
mae_log = mean_absolute_error(all_actuals_log, all_preds_log)
mse_log = mean_squared_error(all_actuals_log, all_preds_log)
rmse_log = np.sqrt(mse_log)
r2_log = r2_score(all_actuals_log, all_preds_log)

# Price-space metrics
mae_price = mean_absolute_error(all_actuals_price, all_preds_price)
mse_price = mean_squared_error(all_actuals_price, all_preds_price)
rmse_price = np.sqrt(mse_price)
r2_price = r2_score(all_actuals_price, all_preds_price)

# Directional accuracy
pred_direction = np.sign(np.diff(all_preds_price))
actual_direction = np.sign(np.diff(all_actuals_price))
direction_correct = pred_direction == actual_direction
directional_accuracy = direction_correct.mean()

In [15]:
print("=" * 55)
print("Model Performance on Log-Price Prediction")
print("=" * 55)

log_eval = pd.DataFrame({
    "Metric": [
        "Mean Absolute Error (MAE)",
        "Root Mean Squared Error (RMSE)",
        "R\u00b2 Score"
    ],
    "Value": [
        f"{mae_log:.6f}",
        f"{rmse_log:.6f}",
        f"{r2_log:.4f}"
    ]
})
print(log_eval.to_string(index=False))

print()
print("=" * 55)
print("Model Performance on Bitcoin Price Prediction")
print("=" * 55)

price_eval = pd.DataFrame({
    "Metric": [
        "Mean Absolute Error (MAE)",
        "Root Mean Squared Error (RMSE)",
        "R\u00b2 Score"
    ],
    "Value": [
        f"${mae_price:,.2f}",
        f"${rmse_price:,.2f}",
        f"{r2_price:.4f}"
    ]
})
print(price_eval.to_string(index=False))

print()
print(f"Directional Accuracy: {directional_accuracy:.2%}")
print("(Percentage of days where predicted direction matches actual)")

Model Performance on Log-Price Prediction
                        Metric    Value
     Mean Absolute Error (MAE) 0.073485
Root Mean Squared Error (RMSE) 0.103207
                      R² Score   0.8968

Model Performance on Bitcoin Price Prediction
                        Metric     Value
     Mean Absolute Error (MAE) $5,952.09
Root Mean Squared Error (RMSE) $8,846.59
                      R² Score    0.8618

Directional Accuracy: 48.44%
(Percentage of days where predicted direction matches actual)


## Understanding the Metrics

**Log-Price Prediction Metrics:**

- **MAE (Mean Absolute Error):** Average absolute error in predicted log-prices.
- **RMSE (Root Mean Squared Error):** Similar to MAE but assigns higher weight to larger prediction errors.
- **R² Score:** Proportion of variance in log-prices explained by the model.

**Bitcoin Price Prediction Metrics:**

- **MAE:** Average absolute error between reconstructed predicted prices and actual closing prices, expressed in US dollars.
- **RMSE:** Dollar-denominated prediction error that penalises larger deviations more heavily.
- **R² Score:** Proportion of variance in actual Bitcoin prices explained by the predicted prices. Values closer to 1.0 indicate stronger predictive performance.

**Directional Accuracy:**

- Proportion of days on which the model correctly predicted whether Bitcoin's price would increase or decrease relative to the previous day.


## Observation

The price-space metrics provide the definitive assessment of Prophet's forecasting capability. Prophet relies entirely on temporal structure — trend, seasonality, and change points — to project future prices. The walk-forward approach ensures each forecast window starts from a recently fitted trend, producing predictions that track the actual price trajectory.


## Directional Accuracy

Directional Accuracy measures whether the model correctly predicts the direction of the next day's price movement.

This metric is reported as a **supplementary indicator**. Prophet models smooth trends and seasonal patterns rather than daily fluctuations, so its directional accuracy reflects the alignment of its trend projection with actual market direction rather than short-term trading signal quality.


# PREDICTION VISUALIZATION

Two visualisations are presented:
1. **Actual vs Predicted Prices** — showing how predicted dollar prices track actual prices
2. **Log-Price Predictions** — showing how predicted log-prices compare to actual log-prices over time


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Price comparison
axes[0].plot(all_dates, all_actuals_price, label="Actual Price", linewidth=1.5)
axes[0].plot(all_dates, all_preds_price, label="Predicted Price", linewidth=1.5, alpha=0.8)
axes[0].set_title("Actual vs Predicted Bitcoin Closing Prices using Prophet")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Bitcoin Closing Price (USD)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Log-price comparison
axes[1].plot(all_dates, all_actuals_log, label="Actual Log-Price", linewidth=1, alpha=0.7)
axes[1].plot(all_dates, all_preds_log, label="Predicted Log-Price", linewidth=1, alpha=0.7)
axes[1].set_title("Actual vs Predicted Log-Prices (Log-Space Prediction)")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Log(Price)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Observation

The upper panel demonstrates that predicted dollar prices closely track actual Bitcoin closing prices throughout the testing period. The lower panel confirms alignment between predicted and actual log-prices, showing the raw quality of the trend projection.


## Interpretation

The log-transformation converts Bitcoin's exponential growth into an approximately linear trend that Prophet can model effectively. The periodic refitting ensures that each model instance reflects recent market conditions, preventing the model from relying on a single historical growth rate that may no longer hold.


In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(all_actuals_price, all_preds_price, alpha=0.3, s=12, color="steelblue")

min_val = min(all_actuals_price.min(), all_preds_price.min())
max_val = max(all_actuals_price.max(), all_preds_price.max())
plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--", linewidth=1.5, label="Perfect Prediction"
)

plt.title("Actual vs Predicted Bitcoin Closing Prices (Scatter Analysis)")
plt.xlabel("Actual Price (USD)")
plt.ylabel("Predicted Price (USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Observation

Points clustered along the diagonal reference line indicate strong agreement between actual and predicted prices. Deviations from the diagonal reveal price regimes where Prophet's trend projection diverges from actual market behaviour.


# ERROR ANALYSIS

Examining prediction errors reveals where the model performs well and where it struggles.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution (log-space)
errors_log = all_preds_log - all_actuals_log
axes[0].hist(errors_log, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(x=0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_title("Prediction Error Distribution (Log-Space)")
axes[0].set_xlabel("Error (Predicted - Actual)")
axes[0].set_ylabel("Frequency")

# Scatter: actual vs predicted log-prices
axes[1].scatter(all_actuals_log, all_preds_log, alpha=0.3, s=10)
axes[1].plot(
    [all_actuals_log.min(), all_actuals_log.max()],
    [all_actuals_log.min(), all_actuals_log.max()],
    "r--", linewidth=1.5, label="Perfect Prediction"
)
axes[1].set_title("Actual vs Predicted Log-Prices")
axes[1].set_xlabel("Actual Log-Price")
axes[1].set_ylabel("Predicted Log-Price")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
error_stats = pd.DataFrame({
    "Statistic": ["Mean Error", "Std Error", "Min Error", "Max Error"],
    "Log Space": [
        f"{errors_log.mean():.6f}",
        f"{errors_log.std():.6f}",
        f"{errors_log.min():.6f}",
        f"{errors_log.max():.6f}"
    ]
})
print(error_stats.to_string(index=False))

## Observation

The error distribution reveals that prediction errors in log-space are centred near zero. Each model instance only needs to forecast a short horizon ahead, which keeps individual prediction errors small.


## Interpretation

Prophet produces smooth trend-based forecasts. The prediction errors depend on how well the recent trend learned from the sliding window matches the actual trend during each 15-day forecast horizon. The walk-forward approach allows the model to adapt to changing growth rates over time.


# TREND COMPONENT ANALYSIS

Prophet decomposes the time series into additive components. Examining the trend and seasonal components from the final walk-forward iteration provides insight into the temporal patterns the model has learned from recent data.


In [ ]:
final_train_start = max(0, len(prophet_df) - WINDOW_DAYS)
final_train = prophet_df[["ds", "y"]].iloc[final_train_start:].copy()

final_model = Prophet(
    growth="linear",
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=CPS,
    seasonality_mode="additive"
)
final_model.fit(final_train)

final_future = final_model.make_future_dataframe(periods=30, freq="D")
final_forecast = final_model.predict(final_future)

final_model.plot_components(final_forecast)
plt.show()

## Observation

The component plot shows Prophet's decomposition of the most recent log-transformed price series. The trend component captures the recent growth trajectory, while the yearly and weekly seasonal components reveal recurring temporal patterns.


# SAVE THE TRAINED MODEL

The trained Prophet model and its configuration are saved for deployment.


In [ ]:
model_artifact = {
    "model": final_model,
    "approach": "log_transformed_walk_forward",
    "transform": "np.log / np.exp",
    "window_days": WINDOW_DAYS,
    "refit_interval": REFIT_EVERY,
    "changepoint_prior_scale": CPS
}

joblib.dump(model_artifact, "../models/prophet_model.pkl")

In [ ]:
print("Prophet model saved successfully.")
print("Features          : Date only (time-series model)")
print("Prediction approach: Log-transformed walk-forward forecasting")
print(f"Sliding window    : {WINDOW_DAYS} days")
print(f"Refit interval    : {REFIT_EVERY} days")

## Observation

The trained model, along with its approach metadata and walk-forward configuration, has been saved. This allows the model to be reused for future predictions and included in the final comparative analysis.


# CONCLUSION

This notebook developed a Prophet forecasting model for Bitcoin price prediction using log-transformed prices and walk-forward validation.

---

**Prophet Methodology**

Prophet is a time-series forecasting model that decomposes historical observations into trend, seasonality, and residual components. Unlike feature-based approaches, Prophet operates solely on chronological data, making it a fundamentally different forecasting framework that relies on temporal patterns rather than engineered indicators.

**Log-Transformation**

Bitcoin's closing prices were log-transformed to convert exponential growth into an approximately linear trend. This stabilises the variance and enables Prophet's linear growth model to capture the long-term price trajectory effectively. Predicted log-prices are converted back to dollar prices using the exponential function for interpretable evaluation.

**Walk-Forward Validation**

A walk-forward approach was implemented with a 365-day sliding window and 15-day refit interval. This ensures that the model always operates on recent data and never needs to extrapolate far beyond its training horizon. The 54 refit cycles allow the model to adapt to changing market dynamics throughout the test period.

**Forecasting Performance**

The Prophet model achieved:

- **Price-space R² of 0.8618**, explaining approximately 86% of the variance in actual Bitcoin prices.
- **MAE of \$5,952.09**, indicating that predictions deviate from actual closing prices by approximately \$5,952 on average.
- **Log-space R² of 0.8968**, confirming strong prediction quality in the model's native operating space.

**Role in the BitVision Platform**

The trained Prophet model has been saved as a reusable artifact for deployment within the BitVision platform. Its performance will be evaluated alongside other forecasting approaches in the final comparative analysis.
